In [2]:
from chromadb.config import Settings
import chromadb

client = chromadb.Client(
    Settings(
        persist_directory="db"
    )
)

In [3]:
# collection = client.create_collection(name="Students")
collection = client.get_or_create_collection(
    name="Students"
)


In [4]:
student_info = """
Alexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,
is a member of the programming and chess clubs who enjoys pizza, swimming, and hiking
in her free time in hopes of working at a tech company after graduating from the University of Washington.
"""

club_info = """
The university chess club provides an outlet for students to come together and enjoy playing
the classic strategy game of chess. Members of all skill levels are welcome, from beginners learning
the rules to experienced tournament players. The club typically meets a few times per week to play casual games,
participate in tournaments, analyze famous chess matches, and improve members' skills.
"""

university_info = """
The University of Washington, founded in 1861 in Seattle, is a public research university
with over 45,000 students across three campuses in Seattle, Tacoma, and Bothell.
As the flagship institution of the six public universities in Washington state,
UW encompasses over 500 buildings and 20 million square feet of space,
including one of the largest library systems in the world.
"""

In [5]:
collection.add(
    documents = [student_info, club_info, university_info],
    metadatas = [{"source": "student info"},{"source": "club info"},{'source':'university info'}],
    ids = ["id1", "id2", "id3"]
)

In [6]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2
)

results

{'ids': [['id1', 'id2']],
 'embeddings': None,
 'documents': [['\nAlexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,\nis a member of the programming and chess clubs who enjoys pizza, swimming, and hiking\nin her free time in hopes of working at a tech company after graduating from the University of Washington.\n',
   "\nThe university chess club provides an outlet for students to come together and enjoy playing\nthe classic strategy game of chess. Members of all skill levels are welcome, from beginners learning\nthe rules to experienced tournament players. The club typically meets a few times per week to play casual games,\nparticipate in tournaments, analyze famous chess matches, and improve members' skills.\n"]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'student info'}, {'source': 'club info'}]],
 'distances': [[1.2946667671203613, 1.3954033851623535]]}

In [7]:
from sentence_transformers import SentenceTransformer

class SentenceTransformerEmbedding:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self._model_name = model_name

    def __call__(self, input):
        return self.model.encode(input).tolist()

    def embed_query(self, input):
        return self.model.encode(input).tolist()

    def name(self):
        return f"sentence-transformers-{self._model_name}"

    def get_config(self):
        return {"model_name": self._model_name}

In [8]:
embedding_function = SentenceTransformerEmbedding()

collection2 = client.get_or_create_collection(
    name="stu_docs",
    embedding_function=embedding_function
)

In [9]:
import inspect
inspect.signature(SentenceTransformerEmbedding.__call__)


<Signature (self, input)>

In [10]:
student_record = """
Student Name: Alexandra Thompson
Student ID: 219584
Age: 19
Academic Year: Sophomore
Major: Computer Science
GPA: 3.7
Enrollment Status: Full-time
Institution: University of Washington
Academic Standing: Good
"""

advising_notes = """
Advising Session Summary – Fall Quarter
Alexandra met with her academic advisor to review degree progress and
discuss potential internship opportunities in the tech industry.
She is on track with core computer science requirements and was advised
to begin preparing for summer internship applications and technical interviews.
"""

student_activities = """
Student Involvement Record
Clubs and Organizations: Programming Club, Chess Club
Recreational Interests: Swimming, hiking
Additional Notes: Demonstrates strong analytical skills and enjoys
collaborative problem-solving activities through club participation.
"""


In [11]:
collection2.add(
    documents = [student_record, advising_notes, student_activities],
    metadatas = [{"source": "student record"},{"source": "advising notes"},{'source':'student activities'}],
    ids = ["id1", "id2", "id3"]
)

In [12]:
results = collection2.query(
    query_texts=["What is the student name?"],
    n_results=2
)

results

{'ids': [['id1', 'id3']],
 'embeddings': None,
 'documents': [['\nStudent Name: Alexandra Thompson\nStudent ID: 219584\nAge: 19\nAcademic Year: Sophomore\nMajor: Computer Science\nGPA: 3.7\nEnrollment Status: Full-time\nInstitution: University of Washington\nAcademic Standing: Good\n',
   '\nStudent Involvement Record\nClubs and Organizations: Programming Club, Chess Club\nRecreational Interests: Swimming, hiking\nAdditional Notes: Demonstrates strong analytical skills and enjoys\ncollaborative problem-solving activities through club participation.\n']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'student record'},
   {'source': 'student activities'}]],
 'distances': [[0.9241572618484497, 1.2298921346664429]]}

In [14]:
print(collection2.count())


3


In [15]:
print(collection.peek())


3
{'ids': ['id1', 'id2', 'id3'], 'embeddings': array([[-0.03421865,  0.01539667, -0.01704109, ..., -0.05819754,
        -0.04708823, -0.102203  ],
       [ 0.02532907, -0.02509279, -0.00254029, ...,  0.03525017,
        -0.11900325, -0.06710909],
       [ 0.0875465 ,  0.01729166, -0.04267626, ..., -0.00880426,
         0.01187566,  0.02228451]]), 'documents': ['\nAlexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,\nis a member of the programming and chess clubs who enjoys pizza, swimming, and hiking\nin her free time in hopes of working at a tech company after graduating from the University of Washington.\n', "\nThe university chess club provides an outlet for students to come together and enjoy playing\nthe classic strategy game of chess. Members of all skill levels are welcome, from beginners learning\nthe rules to experienced tournament players. The club typically meets a few times per week to play casual games,\nparticipate in tournaments, analyze famous ch

In [16]:
client.list_collections()

[Collection(name=Students), Collection(name=stu_docs)]